# SWNA Models — MSD Overlay Comparison

This notebook shows, for every fully implemented SWNA model, how the shape of the theoretical MSD curve changes as each parameter is varied — with all curves for a given parameter overlaid on one set of axes.

Unlike the main workshop notebook's interactive sliders (one curve at a time), the purpose here is purely comparative: static panels, several parameter values per panel, so the effect of each parameter is visible at a glance.

**Scope.** Only the models with working formulas in the `whitenoise` package are covered:
- `exponential` (Table 3.1, row 4)
- `cosine` (Table 3.1, row 10)
- `sine` (Table 3.1, row 9)
- `fbm` (Table 3.1, row 1)
- `dna` (workshop addition, restricted-diffusion form, Violanda et al. 2019)

The remaining 12 models in the package's `MODELS` registry (`exp_whittaker`, `bessel_K`, `hypergeom_F1`, `bessel_I`, `sin_half`, `cos_half`, `hypergeom_3F2`, `csc_power`, `cot_power`, `inc_gamma`, `bessel_pair`, `bessel_pair2`) are stubs — they raise `NotImplementedError` and have no formula to plot. They are intentionally omitted rather than hand-copied from the literature, so this notebook never drifts out of sync with the package.

---

In [ ]:
!pip install git+https://github.com/pnayga/whitenoise.git -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from whitenoise.core.models import msd_exponential, msd_cosine, msd_sine, msd_fbm, MODELS
msd_dna = MODELS['dna']['msd']

# Lag axes. T_cmp's nu-upper-bound (0.03) keeps nu*T/2 well below pi/2 over this
# range, so cosine/sine stay finite (no cos/sin <= 0 cutoff) across every overlay below.
T_cmp = np.linspace(0.5, 80, 400)
L_cmp = np.linspace(1, 2000, 500)

# Ordered low->high palette: same hue progression reused across every panel below,
# so "leftmost/bluest = smallest value" reads consistently model to model.
COLORS5 = ['#1B3A6B', '#2E86AB', '#27AE60', '#F39C12', '#C0392B']

def _valid(msd):
    return np.isfinite(msd) & (msd > 0)

---
## Exponential model — `msd_exponential(T, μ, β)`

MSD(T) = Γ(μ)·β^(−μ)·T^(μ−1)·e^(−β/T)

μ classification here is memory-strength (via the exponent (μ−1)/2), not a diffusive-regime label: μ=1 memoryless, μ>1 long memory, μ<1 short memory (Tey et al. 2024). Published benchmarks: earthquakes μ≈1.00–1.19 (Roque et al. 2024), sunspots μ≈1.15 (Toledo et al. 2024).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Exponential model — parameter overlay', fontsize=13, fontweight='bold')

ax = axes[0]
MU_VALS = [0.5, 1.0, 1.5, 2.0, 3.0]
for mu, col in zip(MU_VALS, COLORS5):
    msd = msd_exponential(T_cmp, mu, 0.1)
    ok = _valid(msd)
    ax.plot(T_cmp[ok], msd[ok], color=col, lw=2.2, label=f'μ={mu}')
ax.set_title('Varying μ  (β = 0.1 fixed)', fontsize=10)
ax.set_xlabel('Lag T'); ax.set_ylabel('MSD')
ax.legend(fontsize=8); ax.grid(alpha=0.2, linestyle='--')

ax = axes[1]
BETA_VALS = [0.02, 0.05, 0.1, 0.3, 0.8]
for beta, col in zip(BETA_VALS, COLORS5):
    msd = msd_exponential(T_cmp, 1.15, beta)
    ok = _valid(msd)
    ax.plot(T_cmp[ok], msd[ok], color=col, lw=2.2, label=f'β={beta}')
ax.set_title('Varying β  (μ = 1.15, published sunspot benchmark)', fontsize=10)
ax.set_xlabel('Lag T'); ax.set_ylabel('MSD')
ax.legend(fontsize=8); ax.grid(alpha=0.2, linestyle='--')

plt.tight_layout()
plt.show()

---
## Cosine model — `msd_cosine(T, μ, ν)`

MSD(T) = √π·Γ(μ)·cos(νT/2)·Jₓ(νT/2) / (T/ν)^(1/2−μ)

Short-time reduction gives MSD ≈ c·T^α with α = 2μ−1: α<1 subdiffusive, α=1 Brownian, 1<α<2 superdiffusive, α>2 hyperballistic (Elnar et al. 2021). Published benchmark: GBR coral μ≈4.64 (Elnar et al. 2021).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Cosine model — parameter overlay', fontsize=13, fontweight='bold')

ax = axes[0]
MU_VALS = [0.5, 1.0, 1.5, 2.0, 2.8]
for mu, col in zip(MU_VALS, COLORS5):
    msd = msd_cosine(T_cmp, mu, 0.008)
    ok = _valid(msd)
    ax.plot(T_cmp[ok], msd[ok], color=col, lw=2.2, label=f'μ={mu}')
ax.set_title('Varying μ  (ν = 0.008 fixed)', fontsize=10)
ax.set_xlabel('Lag T'); ax.set_ylabel('MSD')
ax.legend(fontsize=8); ax.grid(alpha=0.2, linestyle='--')

ax = axes[1]
NU_VALS = [0.003, 0.008, 0.015, 0.02, 0.03]
for nu, col in zip(NU_VALS, COLORS5):
    msd = msd_cosine(T_cmp, 1.2, nu)
    ok = _valid(msd)
    ax.plot(T_cmp[ok], msd[ok], color=col, lw=2.2, label=f'ν={nu}')
ax.set_title('Varying ν  (μ = 1.2 fixed)', fontsize=10)
ax.set_xlabel('Lag T'); ax.set_ylabel('MSD')
ax.legend(fontsize=8); ax.grid(alpha=0.2, linestyle='--')

plt.tight_layout()
plt.show()

---
## Sine model — `msd_sine(T, μ, ν)`

Same structure as the cosine model with sin(νT/2) in place of cos(νT/2); same α = 2μ−1 diffusive-regime reduction applies (Elnar et al. 2021).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Sine model — parameter overlay', fontsize=13, fontweight='bold')

ax = axes[0]
MU_VALS = [0.5, 1.0, 1.5, 2.0, 2.8]
for mu, col in zip(MU_VALS, COLORS5):
    msd = msd_sine(T_cmp, mu, 0.008)
    ok = _valid(msd)
    ax.plot(T_cmp[ok], msd[ok], color=col, lw=2.2, label=f'μ={mu}')
ax.set_title('Varying μ  (ν = 0.008 fixed)', fontsize=10)
ax.set_xlabel('Lag T'); ax.set_ylabel('MSD')
ax.legend(fontsize=8); ax.grid(alpha=0.2, linestyle='--')

ax = axes[1]
NU_VALS = [0.003, 0.008, 0.015, 0.02, 0.03]
for nu, col in zip(NU_VALS, COLORS5):
    msd = msd_sine(T_cmp, 1.2, nu)
    ok = _valid(msd)
    ax.plot(T_cmp[ok], msd[ok], color=col, lw=2.2, label=f'ν={nu}')
ax.set_title('Varying ν  (μ = 1.2 fixed)', fontsize=10)
ax.set_xlabel('Lag T'); ax.set_ylabel('MSD')
ax.legend(fontsize=8); ax.grid(alpha=0.2, linestyle='--')

plt.tight_layout()
plt.show()

---
## fBm model — `msd_fbm(T, H)`

MSD(T) = T^(2H). H=0.5 is exactly Brownian motion (MSD=T); this is a plain mathematical fact and does not carry the subdiffusive/superdiffusive vocabulary used for the cosine/sine α-reduction above.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
fig.suptitle('fBm model — parameter overlay', fontsize=13, fontweight='bold')

H_VALS = [0.2, 0.35, 0.5, 0.65, 0.85]
for H, col in zip(H_VALS, COLORS5):
    msd = msd_fbm(T_cmp, H)
    ax.plot(T_cmp, msd, color=col, lw=2.2, label=f'H={H}')
ax.set_title('Varying H  (MSD = T^(2H))', fontsize=10)
ax.set_xlabel('Lag T'); ax.set_ylabel('MSD')
ax.legend(fontsize=9); ax.grid(alpha=0.2, linestyle='--')
plt.tight_layout()
plt.show()

---
## DNA model — `msd_dna(L, a, b, c)`  (workshop addition, not part of the 16-model table)

MSD(L) = a − c·e^(−bL) — restricted diffusion, plateaus at `a` as L grows. Requires c < a. Published values: a≈5.21, b≈0.0024, c≈3.81 (Violanda et al. 2019). No μ parameter, so no memory/regime classification applies.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
fig.suptitle('DNA model — parameter overlay', fontsize=13, fontweight='bold')

ax = axes[0]
A_VALS = [3.0, 5.21, 7.0, 10.0, 13.0]
for a, col in zip(A_VALS, COLORS5):
    msd = msd_dna(L_cmp, a, 0.0024, 2.5)
    ax.plot(L_cmp, msd, color=col, lw=2.0, label=f'a={a}')
ax.set_title('Varying a (plateau)\n(b=0.0024, c=2.5 fixed)', fontsize=10)
ax.set_xlabel('Occurrence number L'); ax.set_ylabel('MSD')
ax.legend(fontsize=8); ax.grid(alpha=0.2, linestyle='--')

ax = axes[1]
B_VALS = [0.0005, 0.0012, 0.0024, 0.006, 0.012]
for b, col in zip(B_VALS, COLORS5):
    msd = msd_dna(L_cmp, 5.21, b, 3.81)
    ax.plot(L_cmp, msd, color=col, lw=2.0, label=f'b={b}')
ax.axhline(5.21, color='#888', lw=0.8, linestyle=':', alpha=0.6, label='plateau = 5.21')
ax.set_title('Varying b (decay rate)\n(a=5.21, c=3.81 fixed, published values)', fontsize=10)
ax.set_xlabel('Occurrence number L'); ax.set_ylabel('MSD')
ax.legend(fontsize=8); ax.grid(alpha=0.2, linestyle='--')

ax = axes[2]
C_VALS = [1.0, 2.5, 3.81, 4.5, 5.0]
for c, col in zip(C_VALS, COLORS5):
    msd = msd_dna(L_cmp, 5.21, 0.0024, c)
    ax.plot(L_cmp, msd, color=col, lw=2.0, label=f'c={c}')
ax.axhline(5.21, color='#888', lw=0.8, linestyle=':', alpha=0.6, label='plateau = 5.21')
ax.set_title('Varying c (amplitude)\n(a=5.21, b=0.0024 fixed)', fontsize=10)
ax.set_xlabel('Occurrence number L'); ax.set_ylabel('MSD')
ax.legend(fontsize=8); ax.grid(alpha=0.2, linestyle='--')

plt.tight_layout()
plt.show()

---
## Notes

- Every curve here is drawn from the same `msd_*` functions the package uses for fitting (`whitenoise.core.models`) — nothing is a hand-copied formula, so these overlays can never silently drift out of sync with the package.
- `N`, the normalization scalar, is omitted throughout (equivalent to `N=1`): it only rescales amplitude and never changes curve shape or the fitted μ/H.
- To see any single curve interactively with a slider instead of a fixed overlay set, use the sliders in `workshop_notebook.ipynb`.